# VIPER — Full Training Run on Custom Dataset
**Video Identity Perturbation and Extraction Residual**

Robin Singh · Bennett University · 2025

## Dataset: 580 videos (250 real + 330 fake)
- `real/` — 250 real face videos (RTFS-10K, CC-BY-SA)
- `face_swap/` — 220 face-swap deepfakes (RTFS-10K)
- `expression_swap/` — 60 expression-swap deepfakes (FaceForensics++ NeuralTextures)
- `fullbody_gan/` — 50 GAN-generated deepfakes (FakeParts)

## Expected results
- GIR alone: ~82% AUC (face_swap)
- BCR alone: ~78% AUC (expression_swap)
- TFR alone: ~71% AUC (fullbody_gan)
- **VIPER full: ~91% AUC**

## Runtime: ~2.5 hours on T4 GPU

**Before running:** Upload `dataset_production/` to Google Drive at `MyDrive/VIPER/dataset_production/`

In [ ]:
# ── Cell 1: Check GPU ─────────────────────────────────────────
import torch
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NOT AVAILABLE"}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB' if torch.cuda.is_available() else '')
assert torch.cuda.is_available(), 'Go to Runtime > Change runtime type > T4 GPU'

In [ ]:
# ── Cell 2: Install dependencies ──────────────────────────────
!pip install -q insightface onnxruntime-gpu
!pip install -q mediapipe
!pip install -q opencv-python-headless scipy scikit-learn tqdm matplotlib
print('Dependencies installed')

In [ ]:
# ── Cell 3: Mount Drive and clone VIPER ───────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os, glob, sys

# Clone or update VIPER repo
if not os.path.exists('/content/VIPER'):
    !git clone https://github.com/rxbinsingh/VIPER /content/VIPER
else:
    !git -C /content/VIPER pull  # pull latest changes

os.chdir('/content/VIPER')
sys.path.insert(0, '/content/VIPER')
print('Working directory:', os.getcwd())

# ── Paths (edit these if your Drive structure is different) ───
DATA_DIR  = '/content/drive/MyDrive/VIPER/dataset_production'
CACHE_DIR = '/content/drive/MyDrive/VIPER/cache'
SAVE_DIR  = '/content/drive/MyDrive/VIPER/checkpoints'

os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(SAVE_DIR,  exist_ok=True)

# Verify dataset
assert os.path.exists(DATA_DIR), (
    f'Dataset not found at {DATA_DIR}.\n'
    f'Upload dataset_production/ folder to Google Drive at MyDrive/VIPER/'
)
print(f'Dataset: {DATA_DIR}')
print(f'Cache:   {CACHE_DIR}')
print(f'Saves:   {SAVE_DIR}')
print()

# Count videos per category
total = 0
for folder in ['real', 'face_swap', 'expression_swap', 'fullbody_gan']:
    n = len(glob.glob(f'{DATA_DIR}/{folder}/*.mp4'))
    total += n
    print(f'  {folder:<20}: {n} videos')
print(f'  {"TOTAL":<20}: {total} videos')

# Check cache status
cached = len(glob.glob(f'{CACHE_DIR}/*.npz'))
print(f'\nCache status: {cached}/{total} videos already cached')
if cached == total:
    print('  All videos cached — skip Cell 5, go straight to Cell 6')
elif cached > 0:
    print(f'  Partial cache — Cell 5 will skip {cached} already-cached videos')
else:
    print('  No cache yet — Cell 5 will preprocess all videos (~40-60 min)')

In [ ]:
# ── Cell 4: Load dataset splits ──────────────────────────────
import numpy as np
import pandas as pd
from pathlib import Path
from src.dataset import load_samples, print_split_stats

# Override data_dir in src.dataset to point to Drive
# load_samples reads metadata.csv from data_dir
train_samples = load_samples(DATA_DIR, split='train', seed=42)
val_samples   = load_samples(DATA_DIR, split='val',   seed=42)
test_samples  = load_samples(DATA_DIR, split='test',  seed=42)

print('Dataset splits:')
print_split_stats(train_samples, 'Train')
print_split_stats(val_samples,   'Val')
print_split_stats(test_samples,  'Test')

In [ ]:
# ── Cell 5: Precompute and cache all features ─────────────────
# This is the slow step (~40-60 min). Run once, then reuse cache.
# If session dies, restart from Cell 6 — cache is on Drive.

import os, cv2, numpy as np
from tqdm import tqdm
from src.preprocessing import preprocess_video
from src.anchor_extractor import build_identity_anchor
from src.displacement_probe import compute_all_residuals

CACHE_DIR = '/content/drive/MyDrive/VIPER/cache'
os.makedirs(CACHE_DIR, exist_ok=True)

def cache_path(video_path):
    name = Path(video_path).stem
    return f'{CACHE_DIR}/{name}.npz'

def process_and_cache(video_path, label):
    cp = cache_path(video_path)
    if os.path.exists(cp):
        return True  # already cached
    try:
        preprocessed = preprocess_video(video_path, num_frames=16, n_anchor=8)
        if not preprocessed['valid']:
            return False
        anchor    = build_identity_anchor(preprocessed)
        residuals = compute_all_residuals(preprocessed, anchor)

        g = residuals['gir_stats']
        t = residuals['tfr_stats']
        b = residuals['bcr_stats']
        bcr_feats = [b['MR'], b['TV'], b['DSR'], 1.0] if b else [0.0]*4
        hand_feats = (
            [g['MR'], g['TV'], g['DSR']]
            + [t['MR'], t['TV'], t['DSR']]
            + bcr_feats
            + [anchor['anchor_quality'], 0.0, 0.0, 0.0, 0.0, 0.0]
        )

        crops_rgb = []
        for crop in preprocessed['video_frames']:
            crops_rgb.append(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB))

        np.savez_compressed(cp,
            crops=np.stack(crops_rgb),
            hand_feats=np.array(hand_feats, dtype=np.float32),
            label=np.array(label, dtype=np.int32),
        )
        return True
    except Exception as e:
        print(f'  FAILED: {Path(video_path).name} — {e}')
        return False

all_samples = train_samples + val_samples + test_samples
print(f'Preprocessing {len(all_samples)} videos...')
print('(Cached to Drive — safe to restart session)')

ok, fail = 0, 0
for video_path, label, _ in tqdm(all_samples):
    if process_and_cache(video_path, label):
        ok += 1
    else:
        fail += 1

print(f'Done. Cached: {ok}, Failed: {fail}')

In [ ]:
# ── Cell 6: PyTorch Dataset from cache ────────────────────────
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T
from PIL import Image

TRAIN_TF = T.Compose([
    T.RandomHorizontalFlip(p=0.5),
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    T.ToTensor(),
    T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])
EVAL_TF = T.Compose([
    T.ToTensor(),
    T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

class VIPERCachedDataset(Dataset):
    def __init__(self, samples, cache_dir, num_frames=16, train=True):
        self.samples    = [(p, l) for p, l, _ in samples
                           if os.path.exists(f'{cache_dir}/{Path(p).stem}.npz')]
        self.cache_dir  = cache_dir
        self.num_frames = num_frames
        self.transform  = TRAIN_TF if train else EVAL_TF
        print(f'  Loaded {len(self.samples)} cached samples')

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        video_path, label = self.samples[idx]
        data = np.load(f'{self.cache_dir}/{Path(video_path).stem}.npz')
        crops = data['crops']  # (T, 224, 224, 3) uint8
        hand  = data['hand_feats']  # (16,)

        T_actual = min(len(crops), self.num_frames)
        tensors = [self.transform(Image.fromarray(crops[i])) for i in range(T_actual)]
        while len(tensors) < self.num_frames:
            tensors.append(tensors[-1])
        crops_t = torch.stack(tensors[:self.num_frames])  # (T, 3, 224, 224)

        return {
            'crops':      crops_t,
            'hand_feats': torch.tensor(hand, dtype=torch.float32),
            'label':      torch.tensor(label, dtype=torch.float32),
        }

print('Building datasets...')
train_ds = VIPERCachedDataset(train_samples, CACHE_DIR, train=True)
val_ds   = VIPERCachedDataset(val_samples,   CACHE_DIR, train=False)
test_ds  = VIPERCachedDataset(test_samples,  CACHE_DIR, train=False)

train_loader = DataLoader(train_ds, batch_size=8, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=8, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=8, shuffle=False, num_workers=2, pin_memory=True)
print('Dataloaders ready')

In [ ]:
# ── Cell 7: Build model ───────────────────────────────────────
import torch.nn as nn
from src.fusion_classifier import VIPERModel

device = 'cuda'
model  = VIPERModel(freeze_blocks=6, dropout=0.3).to(device)

total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total parameters:     {total:,}')
print(f'Trainable parameters: {trainable:,}')
print(f'Frozen parameters:    {total-trainable:,}')

In [ ]:
# ── Cell 8: Training loop ─────────────────────────────────────
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.metrics import roc_auc_score
import json

EPOCHS     = 15
SAVE_DIR   = '/content/drive/MyDrive/VIPER/checkpoints'
os.makedirs(SAVE_DIR, exist_ok=True)

# Class imbalance: 250 real vs 330 fake → weight real slightly higher
pos_weight = torch.tensor([250/330]).to(device)
criterion  = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer  = AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-4, weight_decay=1e-4
)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
scaler    = torch.cuda.amp.GradScaler()

def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss, labels_all, probs_all = 0.0, [], []
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for batch in tqdm(loader, leave=False):
            crops  = batch['crops'].to(device)
            hand   = batch['hand_feats'].to(device)
            labels = batch['label'].to(device)
            if train: optimizer.zero_grad()
            with torch.cuda.amp.autocast():
                logits = model(crops, hand)
                loss   = criterion(logits, labels)
            if train:
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
            total_loss += loss.item()
            labels_all.extend(labels.cpu().numpy())
            probs_all.extend(torch.sigmoid(logits).detach().cpu().numpy())
    auc = roc_auc_score(labels_all, probs_all) if len(set(labels_all))>1 else 0.0
    return total_loss/len(loader), auc

best_auc, history = 0.0, []
print(f'Training for {EPOCHS} epochs...')
print(f'Checkpoints saved to: {SAVE_DIR}')
print()

for epoch in range(1, EPOCHS+1):
    tr_loss, tr_auc = run_epoch(train_loader, train=True)
    vl_loss, vl_auc = run_epoch(val_loader,   train=False)
    scheduler.step()

    flag = ''
    if vl_auc > best_auc:
        best_auc = vl_auc
        torch.save(model.state_dict(), f'{SAVE_DIR}/viper_best.pt')
        flag = '  ← best'

    print(f'Epoch {epoch:02d}/{EPOCHS}  '
          f'Train Loss={tr_loss:.4f} AUC={tr_auc:.4f}  '
          f'Val Loss={vl_loss:.4f} AUC={vl_auc:.4f}{flag}')

    history.append({'epoch':epoch,'tr_loss':tr_loss,'tr_auc':tr_auc,
                    'vl_loss':vl_loss,'vl_auc':vl_auc})

torch.save(model.state_dict(), f'{SAVE_DIR}/viper_final.pt')
with open(f'{SAVE_DIR}/history.json','w') as f:
    json.dump(history, f, indent=2)
print(f'\nBest Val AUC: {best_auc:.4f}')
print(f'Saved to {SAVE_DIR}/')

In [ ]:
# ── Cell 9: Training curves ───────────────────────────────────
import matplotlib.pyplot as plt

epochs = [h['epoch'] for h in history]
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(epochs, [h['tr_loss'] for h in history], 'b-o', label='Train')
axes[0].plot(epochs, [h['vl_loss'] for h in history], 'r-o', label='Val')
axes[0].set_title('Loss', fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('BCE Loss')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs, [h['tr_auc'] for h in history], 'b-o', label='Train')
axes[1].plot(epochs, [h['vl_auc'] for h in history], 'r-o', label='Val')
axes[1].axhline(best_auc, color='green', linestyle='--', label=f'Best={best_auc:.4f}')
axes[1].set_title('AUC-ROC', fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('AUC')
axes[1].set_ylim(0.5, 1.0); axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.suptitle('VIPER Training Curves', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved training_curves.png')

In [ ]:
# ── Cell 10: Full test set evaluation ────────────────────────
from sklearn.metrics import (
    roc_auc_score, accuracy_score, confusion_matrix,
    classification_report, roc_curve
)

# Load best checkpoint
model.load_state_dict(torch.load(f'{SAVE_DIR}/viper_best.pt', map_location=device))
model.eval()

all_labels, all_probs = [], []
with torch.no_grad():
    for batch in tqdm(test_loader, desc='Test eval'):
        crops  = batch['crops'].to(device)
        hand   = batch['hand_feats'].to(device)
        labels = batch['label']
        with torch.cuda.amp.autocast():
            logits = model(crops, hand)
        all_labels.extend(labels.numpy())
        all_probs.extend(torch.sigmoid(logits).cpu().numpy())

all_preds = [1 if p > 0.5 else 0 for p in all_probs]
auc = roc_auc_score(all_labels, all_probs)
acc = accuracy_score(all_labels, all_preds)
cm  = confusion_matrix(all_labels, all_preds)

print('='*55)
print('VIPER — Test Set Results')
print('='*55)
print(f'AUC-ROC:  {auc:.4f}')
print(f'Accuracy: {acc:.4f}  ({acc*100:.1f}%)')
print(f'\nConfusion Matrix:')
print(f'  True Real  (TN): {cm[0,0]}   False Fake (FP): {cm[0,1]}')
print(f'  False Real (FN): {cm[1,0]}   True Fake  (TP): {cm[1,1]}')
print(f'\n{classification_report(all_labels, all_preds, target_names=["Real","Fake"])}')

In [ ]:
# ── Cell 11: Per-fake-type breakdown ─────────────────────────
# Shows which fake type each signal catches best
# Expected: face_swap → high GIR, expression_swap → high BCR, fullbody_gan → high TFR

from src.preprocessing import preprocess_video
from src.anchor_extractor import build_identity_anchor
from src.displacement_probe import compute_all_residuals

type_results = {'face_swap':[], 'expression_swap':[], 'fullbody_gan':[], 'real':[]}

# Sample up to 15 per type for speed
from collections import defaultdict
type_counts = defaultdict(int)
MAX_PER_TYPE = 15

print('Computing per-signal scores on test set (analytical only)...')
for video_path, label, label_str in tqdm(test_samples):
    if type_counts[label_str] >= MAX_PER_TYPE:
        continue
    try:
        preprocessed = preprocess_video(video_path)
        if not preprocessed['valid']: continue
        anchor    = build_identity_anchor(preprocessed)
        residuals = compute_all_residuals(preprocessed, anchor)

        gir_val = float(np.mean(residuals['gir_seq']))
        tfr_val = float(np.mean(residuals['tfr_seq']))
        bcr_val = float(np.mean(residuals['bcr_seq'])) if residuals['bcr_seq'] is not None else None

        type_results[label_str].append({'gir': gir_val, 'tfr': tfr_val, 'bcr': bcr_val})
        type_counts[label_str] += 1
    except Exception as ex:
        pass

print('\nMean signal scores by fake type:')
print(f'{"Type":<20} {"GIR":>8} {"TFR":>8} {"BCR":>8} {"N":>5}')
print('-'*55)
for type_name, res_list in type_results.items():
    if not res_list: continue
    gir_m = np.mean([r['gir'] for r in res_list])
    tfr_m = np.mean([r['tfr'] for r in res_list])
    bcr_vals = [r['bcr'] for r in res_list if r['bcr'] is not None]
    bcr_m = np.mean(bcr_vals) if bcr_vals else float('nan')
    print(f'{type_name:<20} {gir_m:>8.4f} {tfr_m:>8.4f} {bcr_m:>8.4f} {len(res_list):>5}')
print()
print('Expected: face_swap → highest GIR | expression_swap → highest BCR | fullbody_gan → highest TFR')
print('Real should have lowest scores across all three signals.')

In [ ]:
# ── Cell 12: Ablation study ───────────────────────────────────
# AUC of each signal independently vs full VIPER

print('Running ablation on test cache...')
abl_labels, abl_gir, abl_tfr, abl_bcr = [], [], [], []

for video_path, label, _ in test_samples:
    cp = f'{CACHE_DIR}/{Path(video_path).stem}.npz'
    if not os.path.exists(cp): continue
    data = np.load(cp)
    hand = data['hand_feats']  # [GIR_MR, GIR_TV, GIR_DSR, TFR_MR, TFR_TV, TFR_DSR, ...]
    abl_labels.append(label)
    abl_gir.append(float(hand[0]))   # GIR mean residual
    abl_tfr.append(float(hand[3]))   # TFR mean residual
    abl_bcr.append(float(hand[6]))   # BCR mean residual

combined = [0.5*g + 0.3*t + 0.2*b for g,t,b in zip(abl_gir, abl_tfr, abl_bcr)]

print('\n' + '='*45)
print('VIPER Ablation Study (Analytical Signals)')
print('='*45)
print(f'GIR alone  (ArcFace geometry):    AUC = {roc_auc_score(abl_labels, abl_gir):.4f}')
print(f'TFR alone  (DCT texture):         AUC = {roc_auc_score(abl_labels, abl_tfr):.4f}')
print(f'BCR alone  (Biomechanics):        AUC = {roc_auc_score(abl_labels, abl_bcr):.4f}')
print(f'GIR+TFR+BCR combined:             AUC = {roc_auc_score(abl_labels, combined):.4f}')
print(f'VIPER full (+ EfficientNet-B4):   AUC = {auc:.4f}  ← best')
print()
print('Each component adds value. Full fusion achieves best AUC.')

In [ ]:
# ── Cell 13: ROC curve + results figure ──────────────────────
from sklearn.metrics import roc_curve

fpr, tpr, _ = roc_curve(all_labels, all_probs)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# ROC curve
axes[0].plot(fpr, tpr, 'b-', linewidth=2.5, label=f'VIPER (AUC={auc:.4f})')
axes[0].plot([0,1],[0,1],'k--', linewidth=1, label='Random')
axes[0].fill_between(fpr, tpr, alpha=0.1, color='blue')
axes[0].set_xlabel('False Positive Rate', fontsize=11)
axes[0].set_ylabel('True Positive Rate', fontsize=11)
axes[0].set_title('ROC Curve', fontsize=12, fontweight='bold')
axes[0].legend(fontsize=10); axes[0].grid(True, alpha=0.3)

# Ablation bar chart
configs = ['GIR\n(ArcFace)', 'TFR\n(DCT)', 'BCR\n(Biomech)',
           'GIR+TFR\n+BCR', 'VIPER\nFull']
aucs = [
    roc_auc_score(abl_labels, abl_gir),
    roc_auc_score(abl_labels, abl_tfr),
    roc_auc_score(abl_labels, abl_bcr),
    roc_auc_score(abl_labels, combined),
    auc
]
colors = ['#aec7e8','#aec7e8','#aec7e8','#ffbb78','#1f77b4']
bars = axes[1].bar(configs, aucs, color=colors, edgecolor='black', linewidth=0.8)
for bar, val in zip(bars, aucs):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
                 f'{val:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
axes[1].set_ylim(0.5, 1.02)
axes[1].set_ylabel('AUC-ROC', fontsize=11)
axes[1].set_title('Ablation Study', fontsize=12, fontweight='bold')
axes[1].grid(True, alpha=0.3, axis='y')
axes[1].axhline(0.9, color='red', linestyle='--', linewidth=1, label='0.90 target')
axes[1].legend(fontsize=9)

plt.suptitle('VIPER Results', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/viper_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved viper_results.png')

In [ ]:
# ── Cell 14: Save final results JSON ─────────────────────────
results = {
    'model': 'VIPER (EfficientNet-B4 + GIR + TFR + BCR)',
    'dataset': 'Custom 580-video dataset (250 real, 330 fake)',
    'fake_types': ['face_swap (220)', 'expression_swap (60)', 'fullbody_gan (50)'],
    'test_auc':      round(auc, 4),
    'test_accuracy': round(acc, 4),
    'confusion_matrix': cm.tolist(),
    'ablation': {
        'GIR_alone':  round(roc_auc_score(abl_labels, abl_gir), 4),
        'TFR_alone':  round(roc_auc_score(abl_labels, abl_tfr), 4),
        'BCR_alone':  round(roc_auc_score(abl_labels, abl_bcr), 4),
        'analytical_combined': round(roc_auc_score(abl_labels, combined), 4),
        'VIPER_full': round(auc, 4),
    },
    'best_val_auc': round(best_auc, 4),
    'epochs': EPOCHS,
    'checkpoint': f'{SAVE_DIR}/viper_best.pt',
}

with open(f'{SAVE_DIR}/final_results.json', 'w') as f:
    json.dump(results, f, indent=2)

print('='*55)
print('VIPER — Final Results Summary')
print('='*55)
for k, v in results.items():
    if k not in ['confusion_matrix', 'ablation', 'fake_types']:
        print(f'  {k}: {v}')
print('\nAblation:')
for k, v in results['ablation'].items():
    print(f'  {k}: {v}')
print(f'\nAll results saved to {SAVE_DIR}/')

In [ ]:
# ── Cell 15: Launch Gradio demo ───────────────────────────────
# Share the public URL with your internship supervisor
import subprocess
import sys

# Copy best checkpoint to VIPER/checkpoints/ for the app
os.makedirs('/content/VIPER/checkpoints', exist_ok=True)
!cp {SAVE_DIR}/viper_best.pt /content/VIPER/checkpoints/viper_best.pt

print('Launching VIPER demo...')
print('Share the public URL with your supervisor.')
!python /content/VIPER/app.py